# 08 · Accelerate — 1×1 GPU

`subprocess.run(["accelerate", "launch", "--num_processes", "1", "torch_distributor_trainer.py", ...])` 형태로 호출. `torch_distributor_trainer.py`의 `__main__`이 argparse로 인자를 받고 `train_fn`을 호출한다. `accelerate launch`도 `RANK`/`WORLD_SIZE` 등을 child에 세팅하므로 `dist.init_process_group("nccl")`이 그대로 동작.

> 1×N은 [`09-launch_accelerator_1xN.ipynb`](09-launch_accelerator_1xN.ipynb), M×N은 [`10-launch_accelerator_MxN.ipynb`](10-launch_accelerator_MxN.ipynb).

**클러스터**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (1× A10G), Workers 0. `accelerate==1.5.2`가 사전 설치.

In [ ]:
%run ./00-setup

## 경로

In [ ]:
import os

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
TRAINER_SCRIPT = os.path.join(SCRIPT_DIR, "torch_distributor_trainer.py")
assert os.path.exists(TRAINER_SCRIPT), TRAINER_SCRIPT
print(f"SCRIPT_DIR={SCRIPT_DIR}")
print(f"TRAINER_SCRIPT={TRAINER_SCRIPT}")

## 1×1 GPU

In [ ]:
import subprocess
import mlflow

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="acc-1x1", log_system_metrics=True) as run:
    run_id = run.info.run_id

cmd = [
    "accelerate", "launch",
    "--num_processes", "1",
    TRAINER_SCRIPT,
    "--run_id", run_id,
    "--db_host", DB_HOST,
    "--db_token", DB_TOKEN,
    "--data_dir", DATA_DIR,
    "--ckpt_path", os.path.join(CKPT_DIR, "acc-1x1.pt"),
    "--n_users", str(cfg["n_users"]),
    "--n_items", str(cfg["n_items"]),
    "--emb_dim", str(cfg["emb_dim"]),
    "--tower_hidden", *[str(x) for x in cfg["tower_hidden"]],
    "--batch_size", str(cfg["batch_size_per_gpu"]),
    "--num_epochs", str(cfg["num_epochs"]),
    "--max_steps_per_epoch", str(cfg["max_steps_per_epoch"]),
    "--patience", str(PATIENCE),
    "--min_delta", str(MIN_DELTA),
    "--topology", "1x1",
    "--script_dir", SCRIPT_DIR,
]
print("running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=SCRIPT_DIR)